In [1]:

import os
import joblib
import numpy as np
import pandas as pd

# Load the existing dataset
dataset_path = "Dataset/cycle_time_data.joblib"

data = joblib.load(dataset_path)

print("Loaded object type:", type(data))

if isinstance(data, pd.DataFrame):
    df = data.copy()
    print("\nDataset shape:", df.shape)
    print("\nColumn names:")
    print(df.columns.tolist())
    print("\nFirst five rows:")
    display(df.head())

elif isinstance(data, dict):
    print("\nAvailable dataset keys:")
    for key, value in data.items():
        print(key, ":", type(value),
              getattr(value, "shape", ""))

else:
    print("\nDataset preview:")
    print(str(data)[:1500])


Loaded object type: <class 'dict'>

Available dataset keys:
df : <class 'pandas.DataFrame'> (995, 18)
Xraw : <class 'numpy.ndarray'> (995, 5)
yraw : <class 'numpy.ndarray'> (995,)
features : <class 'list'> 
train_end : <class 'int'> 
stable : <class 'tuple'> 
drift : <class 'tuple'> 
recovery : <class 'tuple'> 
window : <class 'int'> 
drift_multiplier : <class 'float'> 
recovery_multiplier : <class 'float'> 


In [2]:

# Step 2: Inspect dataset columns and bottleneck labels

df = data["df"].copy()

print("Dataset columns:")
print(df.columns.tolist())

print("\nInput features:")
print(data["features"])

print("\nDataset boundaries:")
for key in ["train_end", "stable", "drift",
            "recovery", "window"]:
    print(f"{key}: {data[key]}")

print("\nPossible bottleneck-related columns:")
related_cols = [
    col for col in df.columns
    if any(word in col.lower()
           for word in ["bottleneck", "label", "target"])
]

print(related_cols)

if related_cols:
    for col in related_cols:
        print(f"\n{col} value counts:")
        print(df[col].value_counts(dropna=False))

print("\nFirst five observations:")
display(df.head())


Dataset columns:
['Job_ID', 'Machine_ID', 'Operation_Type', 'Material_Used', 'Processing_Time', 'Energy_Consumption', 'Machine_Availability', 'Scheduled_Start', 'Scheduled_End', 'Actual_Start', 'Actual_End', 'Job_Status', 'Optimization_Category', 'Station_ID', 'Upstream_Lag', 'PT_orig', 'mult', 'phase']

Input features:
['Processing_Time', 'Energy_Consumption', 'Machine_Availability', 'Upstream_Lag', 'Station_ID']

Dataset boundaries:
train_end: 400
stable: (400, 600)
drift: (600, 800)
recovery: (800, None)
window: 10

Possible bottleneck-related columns:
[]

First five observations:


,Job_ID,Machine_ID,Operation_Type,Material_Used,Processing_Time,Energy_Consumption,Machine_Availability,Scheduled_Start,Scheduled_End,Actual_Start,Actual_End,Job_Status,Optimization_Category,Station_ID,Upstream_Lag,PT_orig,mult,phase
0,J002,M01,Grinding,3.35,79.0,6.61,84,3/18/2023 8:10,3/18/2023 9:29,3/18/2023 8:20,3/18/2023 9:39,Delayed,Low Efficiency,0,76.0,79.0,1.0,train
1,J004,M04,Grinding,1.76,106.0,12.50,95,3/18/2023 8:30,3/18/2023 10:16,3/18/2023 8:35,3/18/2023 10:21,Completed,Moderate Efficiency,1,56.0,106.0,1.0,train
2,J005,M01,Lathe,1.90,46.0,8.13,88,3/18/2023 8:40,3/18/2023 9:26,3/18/2023 8:42,3/18/2023 9:28,Completed,High Efficiency,0,79.0,46.0,1.0,train
3,J007,M04,Milling,4.67,22.0,14.20,87,3/18/2023 9:00,3/18/2023 9:22,3/18/2023 9:04,3/18/2023 9:26,Completed,Moderate Efficiency,1,106.0,22.0,1.0,train
4,J009,M02,Drilling,3.49,42.0,8.97,81,3/18/2023 9:20,3/18/2023 10:02,3/18/2023 9:23,3/18/2023 10:05,Completed,Moderate Efficiency,2,100.0,42.0,1.0,train


In [3]:

# Step 3: Inspect variables for bottleneck definition

columns = [
    "Processing_Time",
    "Machine_Availability",
    "Upstream_Lag",
    "Station_ID"
]

print("Summary statistics:")
display(df[columns].describe(include="all"))

print("\nMissing values:")
print(df[columns].isna().sum())

print("\nProcessing time by operating phase:")
display(
    df.groupby("phase")["Processing_Time"]
      .agg(["count", "mean", "median", "std", "min", "max"])
      .round(2)
)

print("\nMachine availability by operating phase:")
display(
    df.groupby("phase")["Machine_Availability"]
      .agg(["count", "mean", "median", "std", "min", "max"])
      .round(2)
)

print("\nUpstream lag by operating phase:")
display(
    df.groupby("phase")["Upstream_Lag"]
      .agg(["count", "mean", "median", "std", "min", "max"])
      .round(2)
)

print("\nStation counts:")
print(df["Station_ID"].value_counts())


Summary statistics:


,Processing_Time,Machine_Availability,Upstream_Lag,Station_ID
count,995.000000,995.000000,995.000000,995.000000
mean,82.404472,89.155779,82.178392,1.941709
std,38.486169,5.765963,38.311709,1.408018
min,20.000000,80.000000,20.000000,0.000000
25%,52.000000,84.000000,52.000000,1.000000
50%,79.000000,89.000000,78.200000,2.000000
75%,106.000000,94.000000,106.000000,3.000000
max,192.000000,99.000000,192.000000,4.000000



Missing values:
Processing_Time         0
Machine_Availability    0
Upstream_Lag            0
Station_ID              0
dtype: int64

Processing time by operating phase:


,count,mean,median,std,min,max
phase,,,,,,
drift,200,117.46,119.20,48.37,32.0,192.0
recovery,195,84.07,86.25,31.64,23.0,138.0
stable,200,71.23,70.00,28.06,21.0,120.0
train,400,69.66,70.50,28.30,20.0,120.0



Machine availability by operating phase:


,count,mean,median,std,min,max
phase,,,,,,
drift,200,89.14,89.5,5.75,80,99
recovery,195,89.94,90.0,5.76,80,99
stable,200,88.80,89.0,5.30,80,99
train,400,88.96,89.0,5.98,80,99



Upstream lag by operating phase:


,count,mean,median,std,min,max
phase,,,,,,
drift,200,116.33,118.4,48.63,32.0,192.0
recovery,195,83.90,85.1,31.71,23.0,138.0
stable,200,71.90,71.5,28.15,21.0,120.0
train,400,69.40,70.0,28.11,20.0,120.0



Station counts:
Station_ID
0    211
2    209
1    198
3    192
4    185
Name: count, dtype: int64


In [4]:

# Step 4: Investigate bottleneck-related indicators

import pandas as pd
import numpy as np

# Processing time relative to the stable-phase reference
stable_df = df[df["phase"] == "stable"]

reference_time = stable_df["Processing_Time"].median()

print("Stable-phase median processing time:",
      round(reference_time, 2))

# Candidate indicators for investigation only
df["High_Processing_Time"] = (
    df["Processing_Time"] > reference_time * 1.5
)

df["High_Upstream_Lag"] = (
    df["Upstream_Lag"] > reference_time * 1.5
)

# Compare indicators across phases
summary = df.groupby("phase")[
    ["High_Processing_Time", "High_Upstream_Lag"]
].mean() * 100

print("\nPercentage of observations exceeding")
print("1.5 times the stable processing-time median:")
display(summary.round(2))

# Examine processing time across stations
station_summary = (
    df.groupby(["phase", "Station_ID"])
      .agg(
          observations=("Processing_Time", "size"),
          mean_processing_time=("Processing_Time", "mean"),
          mean_upstream_lag=("Upstream_Lag", "mean"),
          mean_availability=("Machine_Availability", "mean")
      )
      .round(2)
)

print("\nStation-level comparison:")
display(station_summary)

# Check whether timing information is available
timing_columns = [
    "Scheduled_Start",
    "Scheduled_End",
    "Actual_Start",
    "Actual_End"
]

print("\nTiming columns available:")
print([c for c in timing_columns if c in df.columns])


Stable-phase median processing time: 70.0

Percentage of observations exceeding
1.5 times the stable processing-time median:


,High_Processing_Time,High_Upstream_Lag
phase,,
drift,60.50,59.50
recovery,29.23,29.23
stable,14.50,15.50
train,13.50,13.00



Station-level comparison:


observations  mean_processing_time  mean_upstream_lag  \
phase    Station_ID                                                          
drift    0                     49                117.88             116.11   
         1                     37                120.99             119.05   
         2                     35                127.41             127.41   
         3                     33                116.61             112.82   
         4                     46                107.20             108.45   
recovery 0                     29                 76.34              77.57   
         1                     40                 82.94              81.97   
         2                     52                 86.10              86.74   
         3                     40                 79.98              79.78   
         4                     34                 93.69              92.07   
stable   0                     47                 70.94              71.55   
         1                     38                 68.82              71.13   
         2                     39                 74.95              72.90   
         3                     41                 69.83              71.95   
         4                     35                 71.74              72.03   
train    0                     86                 69.67              69.52   
         1                     83                 71.02              70.34   
         2                     83                 70.39              71.30   
         3                     78                 70.77              70.33   
         4                     70                 65.90              64.87   

                     mean_availability  
phase    Station_ID                     
drift    0                       89.71  
         1                       88.27  
         2                       88.86  
         3                       88.58  
         4                       89.85  
recovery 0                       90.41  
         1                       88.98  
         2                       90.48  
         3                       89.65  
         4                       90.21  
stable   0                       88.70  
         1                       88.55  
         2                       89.54  
         3                       88.34  
         4                       88.91  
train    0                       88.55  
         1                       89.13  
         2                       88.29  
         3                       89.87  
         4                       89.03


Timing columns available:
['Scheduled_Start', 'Scheduled_End', 'Actual_Start', 'Actual_End']


In [5]:

# Step 4: Investigate bottleneck-related indicators

import pandas as pd
import numpy as np

# Processing time relative to the stable-phase reference
stable_df = df[df["phase"] == "stable"]

reference_time = stable_df["Processing_Time"].median()

print("Stable-phase median processing time:",
      round(reference_time, 2))

# Candidate indicators for investigation only
df["High_Processing_Time"] = (
    df["Processing_Time"] > reference_time * 1.5
)

df["High_Upstream_Lag"] = (
    df["Upstream_Lag"] > reference_time * 1.5
)

# Compare indicators across phases
summary = df.groupby("phase")[
    ["High_Processing_Time", "High_Upstream_Lag"]
].mean() * 100

print("\nPercentage of observations exceeding")
print("1.5 times the stable processing-time median:")
display(summary.round(2))

# Examine processing time across stations
station_summary = (
    df.groupby(["phase", "Station_ID"])
      .agg(
          observations=("Processing_Time", "size"),
          mean_processing_time=("Processing_Time", "mean"),
          mean_upstream_lag=("Upstream_Lag", "mean"),
          mean_availability=("Machine_Availability", "mean")
      )
      .round(2)
)

print("\nStation-level comparison:")
display(station_summary)

# Check whether timing information is available
timing_columns = [
    "Scheduled_Start",
    "Scheduled_End",
    "Actual_Start",
    "Actual_End"
]

print("\nTiming columns available:")
print([c for c in timing_columns if c in df.columns])


Stable-phase median processing time: 70.0

Percentage of observations exceeding
1.5 times the stable processing-time median:


,High_Processing_Time,High_Upstream_Lag
phase,,
drift,60.50,59.50
recovery,29.23,29.23
stable,14.50,15.50
train,13.50,13.00



Station-level comparison:


observations  mean_processing_time  mean_upstream_lag  \
phase    Station_ID                                                          
drift    0                     49                117.88             116.11   
         1                     37                120.99             119.05   
         2                     35                127.41             127.41   
         3                     33                116.61             112.82   
         4                     46                107.20             108.45   
recovery 0                     29                 76.34              77.57   
         1                     40                 82.94              81.97   
         2                     52                 86.10              86.74   
         3                     40                 79.98              79.78   
         4                     34                 93.69              92.07   
stable   0                     47                 70.94              71.55   
         1                     38                 68.82              71.13   
         2                     39                 74.95              72.90   
         3                     41                 69.83              71.95   
         4                     35                 71.74              72.03   
train    0                     86                 69.67              69.52   
         1                     83                 71.02              70.34   
         2                     83                 70.39              71.30   
         3                     78                 70.77              70.33   
         4                     70                 65.90              64.87   

                     mean_availability  
phase    Station_ID                     
drift    0                       89.71  
         1                       88.27  
         2                       88.86  
         3                       88.58  
         4                       89.85  
recovery 0                       90.41  
         1                       88.98  
         2                       90.48  
         3                       89.65  
         4                       90.21  
stable   0                       88.70  
         1                       88.55  
         2                       89.54  
         3                       88.34  
         4                       88.91  
train    0                       88.55  
         1                       89.13  
         2                       88.29  
         3                       89.87  
         4                       89.03


Timing columns available:
['Scheduled_Start', 'Scheduled_End', 'Actual_Start', 'Actual_End']


In [6]:

# Step 5: Inspect scheduling and actual timing data

import pandas as pd

timing_cols = [
    "Scheduled_Start",
    "Scheduled_End",
    "Actual_Start",
    "Actual_End"
]

print("Available timing columns:")
print([c for c in timing_cols if c in df.columns])

# Convert timing columns to datetime
for col in timing_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

print("\nMissing datetime values:")
print(df[timing_cols].isna().sum())

# Calculate scheduling delays (minutes)
df["Start_Delay_Min"] = (
    df["Actual_Start"] - df["Scheduled_Start"]
).dt.total_seconds() / 60

df["End_Delay_Min"] = (
    df["Actual_End"] - df["Scheduled_End"]
).dt.total_seconds() / 60

print("\nDelay summary:")
display(
    df[["Start_Delay_Min", "End_Delay_Min"]]
    .describe()
    .round(2)
)

print("\nDelays by operating phase:")
display(
    df.groupby("phase")[
        ["Start_Delay_Min", "End_Delay_Min"]
    ].agg(["mean", "median", "std", "min", "max"])
    .round(2)
)

print("\nFirst 10 records:")
display(
    df[
        [
            "Job_ID",
            "Station_ID",
            "phase",
            "Start_Delay_Min",
            "End_Delay_Min"
        ]
    ].head(10)
)


Available timing columns:
['Scheduled_Start', 'Scheduled_End', 'Actual_Start', 'Actual_End']

Missing datetime values:
Scheduled_Start      0
Scheduled_End        0
Actual_Start       128
Actual_End         128
dtype: int64

Delay summary:


,Start_Delay_Min,End_Delay_Min
count,867.00,867.00
mean,4.45,4.45
std,9.42,9.42
min,-5.00,-5.00
25%,-2.00,-2.00
50%,1.00,1.00
75%,5.00,5.00
max,30.00,30.00



Delays by operating phase:


Start_Delay_Min                          End_Delay_Min                \
                    mean median    std  min   max          mean median    std   
phase                                                                           
drift               6.78    2.0  10.82 -5.0  30.0          6.78    2.0  10.82   
recovery            3.32    1.0   8.61 -5.0  30.0          3.32    1.0   8.61   
stable              5.11    2.0  10.05 -5.0  30.0          5.11    2.0  10.05   
train               3.54    1.0   8.52 -5.0  30.0          3.54    1.0   8.52   

                     
          min   max  
phase                
drift    -5.0  30.0  
recovery -5.0  30.0  
stable   -5.0  30.0  
train    -5.0  30.0


First 10 records:


,Job_ID,Station_ID,phase,Start_Delay_Min,End_Delay_Min
0,J002,0,train,10.0,10.0
1,J004,1,train,5.0,5.0
2,J005,0,train,2.0,2.0
3,J007,1,train,4.0,4.0
4,J009,2,train,3.0,3.0
5,J010,0,train,24.0,24.0
6,J011,0,train,-2.0,-2.0
7,J013,4,train,5.0,5.0
8,J014,1,train,0.0,0.0
9,J015,3,train,4.0,4.0


In [7]:

# Step 6: Validate timing data before defining bottlenecks

import numpy as np
import pandas as pd

# 1. Compare start and end delays
valid = df[
    ["Start_Delay_Min", "End_Delay_Min"]
].dropna()

difference = (
    valid["Start_Delay_Min"] -
    valid["End_Delay_Min"]
)

print("Valid observations:", len(valid))

print("Identical start/end delays:",
      np.isclose(difference, 0).sum())

print("Percentage identical:",
      round(np.isclose(difference, 0).mean() * 100, 2))

print("\nDelay difference statistics:")
print(difference.describe())

# 2. Missing actual timing values by phase
print("\nMissing actual times by phase:")
display(
    df.groupby("phase")[
        ["Actual_Start", "Actual_End"]
    ].apply(lambda x: x.isna().sum())
)

# 3. Job status and missing timing values
print("\nJob status distribution:")
print(df["Job_Status"].value_counts(dropna=False))

print("\nMissing Actual_End by job status:")
display(
    pd.crosstab(
        df["Job_Status"],
        df["Actual_End"].isna(),
        rownames=["Job status"],
        colnames=["Actual end missing"]
    )
)

# 4. Check whether processing time
# is related to actual job duration
actual_duration = (
    df["Actual_End"] - df["Actual_Start"]
).dt.total_seconds() / 60

comparison = pd.DataFrame({
    "Processing_Time": df["Processing_Time"],
    "Actual_Duration_Min": actual_duration
}).dropna()

print("\nProcessing time vs actual duration:")
print(comparison.corr().round(3))

display(comparison.head())


Valid observations: 867
Identical start/end delays: 867
Percentage identical: 100.0

Delay difference statistics:
count    867.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
dtype: float64

Missing actual times by phase:


,Actual_Start,Actual_End
phase,,
drift,31,31
recovery,25,25
stable,22,22
train,50,50



Job status distribution:
Job_Status
Completed    670
Delayed      197
Failed       128
Name: count, dtype: int64

Missing Actual_End by job status:


Actual end missing,False,True
Job status,,
Completed,670,0
Delayed,197,0
Failed,0,128



Processing time vs actual duration:
                     Processing_Time  Actual_Duration_Min
Processing_Time                1.000                0.881
Actual_Duration_Min            0.881                1.000


,Processing_Time,Actual_Duration_Min
0,79.0,79.0
1,106.0,106.0
2,46.0,46.0
3,22.0,22.0
4,42.0,42.0


In [8]:

# Step 7: Examine job outcomes and operational indicators

import pandas as pd
import numpy as np

print("Job status by operating phase:")
status_counts = pd.crosstab(
    df["phase"],
    df["Job_Status"]
)
display(status_counts)

print("\nJob status percentages by phase:")
status_percent = pd.crosstab(
    df["phase"],
    df["Job_Status"],
    normalize="index"
) * 100
display(status_percent.round(2))

print("\nOperational indicators by job status:")
status_summary = (
    df.groupby("Job_Status")
      .agg(
          observations=("Job_ID", "size"),
          mean_processing_time=("Processing_Time", "mean"),
          median_processing_time=("Processing_Time", "median"),
          mean_upstream_lag=("Upstream_Lag", "mean"),
          mean_availability=("Machine_Availability", "mean")
      )
      .round(2)
)
display(status_summary)

print("\nJob status by station:")
display(
    pd.crosstab(
        df["Station_ID"],
        df["Job_Status"]
    )
)

# Candidate adverse-outcome label, for investigation only
df["Adverse_Job_Outcome"] = (
    df["Job_Status"].isin(["Delayed", "Failed"])
).astype(int)

print("\nAdverse job outcome distribution:")
print(df["Adverse_Job_Outcome"].value_counts())

print("\nAdverse outcome rate by phase (%):")
print(
    (df.groupby("phase")["Adverse_Job_Outcome"]
       .mean() * 100).round(2)
)


Job status by operating phase:


Job_Status,Completed,Delayed,Failed
phase,,,
drift,111,58,31
recovery,139,31,25
stable,137,41,22
train,283,67,50



Job status percentages by phase:


Job_Status,Completed,Delayed,Failed
phase,,,
drift,55.50,29.00,15.50
recovery,71.28,15.90,12.82
stable,68.50,20.50,11.00
train,70.75,16.75,12.50



Operational indicators by job status:


,observations,mean_processing_time,median_processing_time,mean_upstream_lag,mean_availability
Job_Status,,,,,
Completed,670,80.50,78.0,80.32,89.12
Delayed,197,86.52,81.0,85.62,89.37
Failed,128,86.04,78.6,86.59,89.02



Job status by station:


Job_Status,Completed,Delayed,Failed
Station_ID,,,
0,151,37,23
1,128,36,34
2,145,43,21
3,123,48,21
4,123,33,29



Adverse job outcome distribution:
Adverse_Job_Outcome
0    670
1    325
Name: count, dtype: int64

Adverse outcome rate by phase (%):
phase
drift       44.50
recovery    28.72
stable      31.50
train       29.25
Name: Adverse_Job_Outcome, dtype: float64


In [9]:

# Step 8: Check chronological ordering and target distribution

import pandas as pd
import numpy as np

df["Scheduled_Start"] = pd.to_datetime(
    df["Scheduled_Start"], errors="coerce"
)

print("Scheduled start missing values:",
      df["Scheduled_Start"].isna().sum())

print("\nChronologically ordered:",
      df["Scheduled_Start"].is_monotonic_increasing)

# Count any backward movements in time
time_diff = df["Scheduled_Start"].diff()

print("Backward time movements:",
      (time_diff < pd.Timedelta(0)).sum())

print("\nFirst and last scheduled times:")
print("First:", df["Scheduled_Start"].iloc[0])
print("Last: ", df["Scheduled_Start"].iloc[-1])

print("\nTarget distribution by phase:")
display(
    pd.crosstab(
        df["phase"],
        df["Adverse_Job_Outcome"],
        margins=True
    )
)

print("\nTarget transitions by phase:")
for phase in ["train", "stable", "drift", "recovery"]:
    y = df.loc[
        df["phase"] == phase,
        "Adverse_Job_Outcome"
    ].to_numpy()

    events = np.sum((y[:-1] == 0) & (y[1:] == 1))

    print(f"{phase}: {events} transitions from 0 to 1")


Scheduled start missing values: 0

Chronologically ordered: True
Backward time movements: 0

First and last scheduled times:
First: 2023-03-18 08:10:00
Last:  2023-03-25 06:30:00

Target distribution by phase:


Adverse_Job_Outcome,0,1,All
phase,,,
drift,111,89,200
recovery,139,56,195
stable,137,63,200
train,283,117,400
All,670,325,995



Target transitions by phase:
train: 84 transitions from 0 to 1
stable: 45 transitions from 0 to 1
drift: 48 transitions from 0 to 1
recovery: 36 transitions from 0 to 1


In [10]:

# Step 9: Prepare leakage-safe LSTM classification sequences

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

FEATURES = data["features"]
WINDOW = 10
TRAIN_END = 400
HORIZONS = [1, 3, 5]

X_raw = df[FEATURES].to_numpy(dtype=np.float32)

y_raw = df["Adverse_Job_Outcome"].to_numpy(
    dtype=np.int64
)

# Fit scaler using training data only
scaler = StandardScaler()
scaler.fit(X_raw[:TRAIN_END])

X_scaled = scaler.transform(X_raw).astype(np.float32)

def create_sequences(X, y, window, horizon):
    X_seq = []
    y_seq = []
    target_indices = []

    # Window ends at time t.
    # Target is at time t + horizon.
    for t in range(window - 1, len(X) - horizon):

        X_seq.append(
            X[t - window + 1:t + 1]
        )

        target_idx = t + horizon

        y_seq.append(y[target_idx])
        target_indices.append(target_idx)

    return (
        np.asarray(X_seq, dtype=np.float32),
        np.asarray(y_seq, dtype=np.int64),
        np.asarray(target_indices, dtype=np.int64)
    )

datasets = {}

for horizon in HORIZONS:

    X_seq, y_seq, indices = create_sequences(
        X_scaled,
        y_raw,
        WINDOW,
        horizon
    )

    train_mask = indices < TRAIN_END
    test_mask = indices >= TRAIN_END

    datasets[horizon] = {
        "X_train": X_seq[train_mask],
        "y_train": y_seq[train_mask],
        "X_test": X_seq[test_mask],
        "y_test": y_seq[test_mask],
        "test_indices": indices[test_mask]
    }

    print(f"\nHorizon t+{horizon}")

    print("Training sequences:",
          datasets[horizon]["X_train"].shape)

    print("Evaluation sequences:",
          datasets[horizon]["X_test"].shape)

    print("Training positive labels:",
          datasets[horizon]["y_train"].sum())

    print("Evaluation positive labels:",
          datasets[horizon]["y_test"].sum())

    print("First evaluation target index:",
          datasets[horizon]["test_indices"][0])

    print("Last evaluation target index:",
          datasets[horizon]["test_indices"][-1])



Horizon t+1
Training sequences: (390, 10, 5)
Evaluation sequences: (595, 10, 5)
Training positive labels: 115
Evaluation positive labels: 208
First evaluation target index: 400
Last evaluation target index: 994

Horizon t+3
Training sequences: (388, 10, 5)
Evaluation sequences: (595, 10, 5)
Training positive labels: 115
Evaluation positive labels: 208
First evaluation target index: 400
Last evaluation target index: 994

Horizon t+5
Training sequences: (386, 10, 5)
Evaluation sequences: (595, 10, 5)
Training positive labels: 115
Evaluation positive labels: 208
First evaluation target index: 400
Last evaluation target index: 994


In [11]:

# Step 10: Train static LSTM classifiers

import os
import random
import numpy as np
import pandas as pd
import torch

from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix
)

SEED = 0
EPOCHS = 30
BATCH_SIZE = 32
LEARNING_RATE = 0.001
HIDDEN_SIZE = 50
THRESHOLD = 0.50

os.makedirs("Results/Bottleneck", exist_ok=True)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

class BottleneckLSTM(nn.Module):

    def __init__(self, input_size=5, hidden_size=50):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.classifier = nn.Linear(hidden_size, 1)

    def forward(self, x):
        output, _ = self.lstm(x)

        last_hidden = output[:, -1, :]

        logits = self.classifier(last_hidden)

        return logits.squeeze(-1)


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

static_models = {}
static_predictions = {}
static_results = []

for horizon in HORIZONS:

    print(f"\nTraining static LSTM for t+{horizon}")

    set_seed(SEED)

    dataset = datasets[horizon]

    X_train = torch.tensor(
        dataset["X_train"], dtype=torch.float32
    )

    y_train = torch.tensor(
        dataset["y_train"], dtype=torch.float32
    )

    train_loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    model = BottleneckLSTM(
        input_size=len(FEATURES),
        hidden_size=HIDDEN_SIZE
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(EPOCHS):

        model.train()
        total_loss = 0.0

        for xb, yb in train_loader:

            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            logits = model(xb)

            loss = criterion(logits, yb)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(xb)

        if (epoch + 1) % 10 == 0:
            average_loss = total_loss / len(X_train)

            print(
                f"Epoch {epoch+1}/{EPOCHS}, "
                f"Loss: {average_loss:.4f}"
            )

    # Evaluate without retraining
    model.eval()

    X_test = torch.tensor(
        dataset["X_test"], dtype=torch.float32
    ).to(device)

    with torch.no_grad():

        logits = model(X_test)

        probabilities = torch.sigmoid(
            logits
        ).cpu().numpy()

    predictions = (
        probabilities >= THRESHOLD
    ).astype(int)

    y_true = dataset["y_test"]

    precision = precision_score(
        y_true, predictions, zero_division=0
    )

    recall = recall_score(
        y_true, predictions, zero_division=0
    )

    f1 = f1_score(
        y_true, predictions, zero_division=0
    )

    accuracy = accuracy_score(
        y_true, predictions
    )

    static_results.append({
        "horizon": horizon,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": accuracy,
        "positive_predictions": int(predictions.sum())
    })

    static_models[horizon] = model

    static_predictions[horizon] = pd.DataFrame({
        "idx": dataset["test_indices"],
        "actual": y_true,
        "probability": probabilities,
        "predicted": predictions
    })

    torch.save(
        model.state_dict(),
        f"Results/Bottleneck/static_lstm_t{horizon}.pth"
    )

    static_predictions[horizon].to_csv(
        f"Results/Bottleneck/static_predictions_t{horizon}.csv",
        index=False
    )

    print("Confusion matrix:")
    print(confusion_matrix(y_true, predictions))

results_df = pd.DataFrame(static_results)

print("\nStatic LSTM results:")
display(results_df.round(4))

results_df.to_csv(
    "Results/Bottleneck/static_lstm_results.csv",
    index=False
)


Device: cpu

Training static LSTM for t+1
Epoch 10/30, Loss: 0.5983
Epoch 20/30, Loss: 0.5962
Epoch 30/30, Loss: 0.5812
Confusion matrix:
[[387   0]
 [208   0]]

Training static LSTM for t+3
Epoch 10/30, Loss: 0.6002
Epoch 20/30, Loss: 0.5937
Epoch 30/30, Loss: 0.5918
Confusion matrix:
[[386   1]
 [208   0]]

Training static LSTM for t+5
Epoch 10/30, Loss: 0.6054
Epoch 20/30, Loss: 0.5907
Epoch 30/30, Loss: 0.5811
Confusion matrix:
[[386   1]
 [208   0]]

Static LSTM results:


,horizon,precision,recall,f1,accuracy,positive_predictions
0,1,0.0,0.0,0.0,0.6504,0
1,3,0.0,0.0,0.0,0.6487,1
2,5,0.0,0.0,0.0,0.6487,1


In [15]:

# Step 11: Diagnose static LSTM predictions

import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score
)

thresholds = [0.30, 0.40, 0.50, 0.60]

diagnostic_results = []

for horizon in HORIZONS:

    prediction_df = static_predictions[horizon]

    y_true = prediction_df["actual"].to_numpy()
    probabilities = prediction_df["probability"].to_numpy()

    print(f"\nHorizon t+{horizon}")

    print("Probability statistics:")
    print(
        pd.Series(probabilities)
        .describe()
        .round(4)
    )

    for threshold in thresholds:

        y_pred = (
            probabilities >= threshold
        ).astype(int)

        diagnostic_results.append({
            "horizon": horizon,
            "threshold": threshold,
            "precision": precision_score(
                y_true, y_pred, zero_division=0
            ),
            "recall": recall_score(
                y_true, y_pred, zero_division=0
            ),
            "f1": f1_score(
                y_true, y_pred, zero_division=0
            ),
            "balanced_accuracy": balanced_accuracy_score(
                y_true, y_pred
            ),
            "positive_predictions": int(y_pred.sum())
        })

diagnostic_df = pd.DataFrame(diagnostic_results)

print("\nThreshold comparison:")
display(diagnostic_df.round(4))

diagnostic_df.to_csv(
    "Results/Bottleneck/static_threshold_diagnostics.csv",
    index=False
)

# Majority-class baseline
y_reference = datasets[1]["y_test"]
majority_predictions = np.zeros_like(y_reference)

print("\nMajority-class baseline accuracy:")
print(
    round(
        np.mean(y_reference == majority_predictions),
        4
    )
)



Horizon t+1
Probability statistics:
count    595.0000
mean       0.2941
std        0.0708
min        0.1369
25%        0.2418
50%        0.2909
75%        0.3473
max        0.4852
dtype: float64

Horizon t+3
Probability statistics:
count    595.0000
mean       0.3079
std        0.0682
min        0.1213
25%        0.2584
50%        0.3065
75%        0.3517
max        0.5447
dtype: float64

Horizon t+5
Probability statistics:
count    595.0000
mean       0.2807
std        0.0695
min        0.1155
25%        0.2344
50%        0.2779
75%        0.3234
max        0.5095
dtype: float64

Threshold comparison:


,horizon,threshold,precision,recall,f1,balanced_accuracy,positive_predictions
0,1,0.3,0.3623,0.4615,0.4059,0.5124,265
1,1,0.4,0.3958,0.0913,0.1484,0.5082,48
2,1,0.5,0.0000,0.0000,0.0000,0.5000,0
3,1,0.6,0.0000,0.0000,0.0000,0.5000,0
4,3,0.3,0.3249,0.4952,0.3924,0.4711,317
5,3,0.4,0.3000,0.0865,0.1343,0.4890,60
6,3,0.5,0.0000,0.0000,0.0000,0.4987,1
7,3,0.6,0.0000,0.0000,0.0000,0.5000,0
8,5,0.3,0.3163,0.3269,0.3215,0.4735,215
9,5,0.4,0.2903,0.0433,0.0753,0.4932,31



Majority-class baseline accuracy:
0.6504


In [14]:

# Step 12: Threshold-independent evaluation

import os
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

auc_results = []

for horizon in HORIZONS:

    pred_df = static_predictions[horizon]

    y_true = pred_df["actual"].to_numpy()
    probabilities = pred_df["probability"].to_numpy()

    prevalence = y_true.mean()

    roc_auc = roc_auc_score(
        y_true, probabilities
    )

    average_precision = average_precision_score(
        y_true, probabilities
    )

    brier = brier_score_loss(
        y_true, probabilities
    )

    auc_results.append({
        "horizon": f"t+{horizon}",
        "ROC_AUC": roc_auc,
        "Average_Precision": average_precision,
        "Positive_Prevalence": prevalence,
        "Brier_Score": brier
    })

auc_df = pd.DataFrame(auc_results)

print("Static LSTM: threshold-independent metrics")
display(auc_df.round(4))

os.makedirs("Results/Bottleneck", exist_ok=True)

auc_df.to_csv(
    "Results/Bottleneck/static_auc_results.csv",
    index=False
)


Static LSTM: threshold-independent metrics


,horizon,ROC_AUC,Average_Precision,Positive_Prevalence,Brier_Score
0,t+1,0.5182,0.3744,0.3496,0.2334
1,t+3,0.4637,0.3382,0.3496,0.2375
2,t+5,0.4663,0.3250,0.3496,0.2411


In [16]:

# Step 13: Logistic regression baseline
# Uses the same input windows and forecast horizons as LSTM

import os
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score
)

os.makedirs("Results/Bottleneck", exist_ok=True)

logistic_results = []
logistic_predictions = {}

for horizon in HORIZONS:

    dataset = datasets[horizon]

    # Flatten each (10, 5) window into 50 features
    X_train = dataset["X_train"].reshape(
        len(dataset["X_train"]), -1
    )
    X_test = dataset["X_test"].reshape(
        len(dataset["X_test"]), -1
    )

    y_train = dataset["y_train"]
    y_test = dataset["y_test"]

    # Class weighting based on training labels only
    model = LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=0
    )

    model.fit(X_train, y_train)

    probabilities = model.predict_proba(X_test)[:, 1]

    # Fixed threshold, not tuned on evaluation data
    predictions = (probabilities >= 0.50).astype(int)

    logistic_results.append({
        "horizon": f"t+{horizon}",
        "ROC_AUC": roc_auc_score(
            y_test, probabilities
        ),
        "Average_Precision": average_precision_score(
            y_test, probabilities
        ),
        "Precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_test, predictions, zero_division=0
        ),
        "Balanced_Accuracy": balanced_accuracy_score(
            y_test, predictions
        )
    })

    logistic_predictions[horizon] = pd.DataFrame({
        "idx": dataset["test_indices"],
        "actual": y_test,
        "probability": probabilities,
        "predicted": predictions
    })

    logistic_predictions[horizon].to_csv(
        f"Results/Bottleneck/logistic_predictions_t{horizon}.csv",
        index=False
    )

logistic_df = pd.DataFrame(logistic_results)

print("Logistic regression baseline results:")
display(logistic_df.round(4))

logistic_df.to_csv(
    "Results/Bottleneck/logistic_baseline_results.csv",
    index=False
)


Logistic regression baseline results:


,horizon,ROC_AUC,Average_Precision,Precision,Recall,F1,Balanced_Accuracy
0,t+1,0.4995,0.3773,0.3636,0.4231,0.3911,0.5126
1,t+3,0.4847,0.3700,0.3298,0.2981,0.3131,0.4862
2,t+5,0.4807,0.3350,0.3106,0.2404,0.2710,0.4768


In [17]:

# Step 14: Train class-weighted LSTM
# Same architecture and sequences as Step 10

import os
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score
)

weighted_models = {}
weighted_predictions = {}
weighted_results = []

for horizon in HORIZONS:

    print(f"\nTraining weighted LSTM: t+{horizon}")

    set_seed(SEED)

    dataset = datasets[horizon]

    X_train = torch.tensor(
        dataset["X_train"], dtype=torch.float32
    )

    y_train = torch.tensor(
        dataset["y_train"], dtype=torch.float32
    )

    loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    model = BottleneckLSTM(
        input_size=len(FEATURES),
        hidden_size=HIDDEN_SIZE
    ).to(device)

    # Calculate positive-class weight from training labels only
    positive_count = y_train.sum().item()
    negative_count = len(y_train) - positive_count

    pos_weight = torch.tensor(
        [negative_count / positive_count],
        dtype=torch.float32,
        device=device
    )

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=pos_weight
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    for epoch in range(EPOCHS):

        model.train()
        total_loss = 0.0

        for xb, yb in loader:

            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(xb)

        if (epoch + 1) % 10 == 0:
            print(
                f"Epoch {epoch+1}/{EPOCHS}, "
                f"Loss: {total_loss/len(X_train):.4f}"
            )

    model.eval()

    X_test = torch.tensor(
        dataset["X_test"], dtype=torch.float32
    ).to(device)

    with torch.no_grad():
        probabilities = torch.sigmoid(
            model(X_test)
        ).cpu().numpy()

    predictions = (
        probabilities >= 0.50
    ).astype(int)

    y_true = dataset["y_test"]

    weighted_results.append({
        "horizon": f"t+{horizon}",
        "ROC_AUC": roc_auc_score(
            y_true, probabilities
        ),
        "Average_Precision": average_precision_score(
            y_true, probabilities
        ),
        "Precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_true, predictions, zero_division=0
        ),
        "Balanced_Accuracy": balanced_accuracy_score(
            y_true, predictions
        ),
        "Positive_Predictions": int(predictions.sum())
    })

    weighted_models[horizon] = model

    weighted_predictions[horizon] = pd.DataFrame({
        "idx": dataset["test_indices"],
        "actual": y_true,
        "probability": probabilities,
        "predicted": predictions
    })

    torch.save(
        model.state_dict(),
        f"Results/Bottleneck/weighted_lstm_t{horizon}.pth"
    )

    weighted_predictions[horizon].to_csv(
        f"Results/Bottleneck/weighted_predictions_t{horizon}.csv",
        index=False
    )

weighted_df = pd.DataFrame(weighted_results)

print("\nClass-weighted LSTM results:")
display(weighted_df.round(4))

weighted_df.to_csv(
    "Results/Bottleneck/weighted_lstm_results.csv",
    index=False
)



Training weighted LSTM: t+1
Epoch 10/30, Loss: 0.9584
Epoch 20/30, Loss: 0.9549
Epoch 30/30, Loss: 0.9059

Training weighted LSTM: t+3
Epoch 10/30, Loss: 0.9614
Epoch 20/30, Loss: 0.9359
Epoch 30/30, Loss: 0.9136

Training weighted LSTM: t+5
Epoch 10/30, Loss: 0.9651
Epoch 20/30, Loss: 0.9322
Epoch 30/30, Loss: 0.8900

Class-weighted LSTM results:


,horizon,ROC_AUC,Average_Precision,Precision,Recall,F1,Balanced_Accuracy,Positive_Predictions
0,t+1,0.5316,0.3742,0.3836,0.5865,0.4639,0.5400,318
1,t+3,0.5000,0.3491,0.3657,0.4712,0.4118,0.5159,268
2,t+5,0.4944,0.3475,0.3365,0.5048,0.4038,0.4850,312


In [18]:

# Step 15: ADWIN drift detection on classification errors

import os
import numpy as np
import pandas as pd

try:
    from river.drift import ADWIN
except ImportError:
    print("River is not installed.")
    print("Run this in a separate cell:")
    print("%pip install river")
    raise

ADWIN_DELTA = 0.002

drift_results = []
drift_detection_records = []

for horizon in HORIZONS:

    pred_df = weighted_predictions[horizon].copy()

    pred_df = pred_df.sort_values("idx").reset_index(drop=True)

    detector = ADWIN(delta=ADWIN_DELTA)

    detections = []

    for row in pred_df.itertuples(index=False):

        error = int(row.actual != row.predicted)

        detector.update(error)

        if detector.drift_detected:
            detections.append(int(row.idx))

    print(f"\nHorizon t+{horizon}")
    print("Detected drift points:", detections)

    drift_results.append({
        "horizon": f"t+{horizon}",
        "detections": len(detections),
        "first_detection": (
            detections[0] if detections else np.nan
        )
    })

    for point in detections:
        drift_detection_records.append({
            "horizon": horizon,
            "detection_idx": point
        })

drift_df = pd.DataFrame(drift_results)

print("\nADWIN detection summary:")
display(drift_df)

os.makedirs("Results/Bottleneck", exist_ok=True)

drift_df.to_csv(
    "Results/Bottleneck/adwin_detection_summary.csv",
    index=False
)

pd.DataFrame(
    drift_detection_records,
    columns=["horizon", "detection_idx"]
).to_csv(
    "Results/Bottleneck/adwin_detection_points.csv",
    index=False
)



Horizon t+1
Detected drift points: []

Horizon t+3
Detected drift points: []

Horizon t+5
Detected drift points: []

ADWIN detection summary:


,horizon,detections,first_detection
0,t+1,0,NaN
1,t+3,0,NaN
2,t+5,0,NaN


In [20]:

# Step 16: Classification performance by operating phase

import os
import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score
)

phase_results = []

for horizon in HORIZONS:

    pred_df = weighted_predictions[horizon].copy()

    pred_df["phase"] = np.select(
        [
            pred_df["idx"] < 600,
            pred_df["idx"] < 800
        ],
        ["stable", "drift"],
        default="recovery"
    )

    pred_df["error"] = (
        pred_df["actual"] != pred_df["predicted"]
    ).astype(int)

    for phase in ["stable", "drift", "recovery"]:

        subset = pred_df[
            pred_df["phase"] == phase
        ]

        y_true = subset["actual"]
        y_pred = subset["predicted"]

        phase_results.append({
            "horizon": f"t+{horizon}",
            "phase": phase,
            "observations": len(subset),
            "positive_rate": y_true.mean(),
            "prediction_error_rate": subset["error"].mean(),
            "precision": precision_score(
                y_true, y_pred, zero_division=0
            ),
            "recall": recall_score(
                y_true, y_pred, zero_division=0
            ),
            "f1": f1_score(
                y_true, y_pred, zero_division=0
            ),
            "balanced_accuracy": balanced_accuracy_score(
                y_true, y_pred
            )
        })

phase_df = pd.DataFrame(phase_results)

print("Class-weighted LSTM performance by phase:")
display(phase_df.round(4))

os.makedirs("Results/Bottleneck", exist_ok=True)

phase_df.to_csv(
    "Results/Bottleneck/weighted_lstm_phase_results.csv",
    index=False
)


Class-weighted LSTM performance by phase:


,horizon,phase,observations,positive_rate,prediction_error_rate,precision,recall,f1,balanced_accuracy
0,t+1,stable,200,0.3150,0.5300,0.2991,0.5079,0.3765,0.4802
1,t+1,drift,200,0.4450,0.4600,0.4887,0.7303,0.5856,0.5589
2,t+1,recovery,195,0.2872,0.4308,0.3205,0.4464,0.3731,0.5326
3,t+3,stable,200,0.3150,0.4750,0.3400,0.5397,0.4172,0.5290
4,t+3,drift,200,0.4450,0.5000,0.4466,0.5169,0.4792,0.5017
5,t+3,recovery,195,0.2872,0.4359,0.2769,0.3214,0.2975,0.4916
6,t+5,stable,200,0.3150,0.5150,0.2872,0.4286,0.3439,0.4698
7,t+5,drift,200,0.4450,0.5800,0.4082,0.6742,0.5085,0.4452
8,t+5,recovery,195,0.2872,0.4667,0.2535,0.3214,0.2835,0.4701


In [21]:

# Step 17: Compare ADWIN monitoring signals
# Diagnostic experiment: no retraining

import os
import numpy as np
import pandas as pd
from river.drift import ADWIN

ADWIN_DELTA = 0.002

signal_results = []
signal_detections = []

for horizon in HORIZONS:

    pred_df = (
        weighted_predictions[horizon]
        .sort_values("idx")
        .reset_index(drop=True)
    )

    y_true = pred_df["actual"].to_numpy(dtype=int)
    prob = pred_df["probability"].to_numpy(dtype=float)
    y_pred = pred_df["predicted"].to_numpy(dtype=int)
    indices = pred_df["idx"].to_numpy(dtype=int)

    # Prevent log(0)
    p = np.clip(prob, 1e-7, 1 - 1e-7)

    signals = {
        "Binary_error": (y_true != y_pred).astype(float),
        "Brier_loss": (y_true - prob) ** 2,
        "Log_loss": -(
            y_true * np.log(p) +
            (1 - y_true) * np.log(1 - p)
        )
    }

    for signal_name, values in signals.items():

        detector = ADWIN(delta=ADWIN_DELTA)
        detections = []

        for idx, value in zip(indices, values):

            detector.update(float(value))

            if detector.drift_detected:
                detections.append(int(idx))

        signal_results.append({
            "horizon": f"t+{horizon}",
            "signal": signal_name,
            "detections": len(detections),
            "first_detection": (
                detections[0] if detections else np.nan
            ),
            "detection_points": str(detections)
        })

        for idx in detections:
            signal_detections.append({
                "horizon": horizon,
                "signal": signal_name,
                "detection_idx": idx
            })

signal_df = pd.DataFrame(signal_results)

print("ADWIN monitoring-signal comparison:")
display(signal_df)

os.makedirs("Results/Bottleneck", exist_ok=True)

signal_df.to_csv(
    "Results/Bottleneck/adwin_signal_comparison.csv",
    index=False
)

pd.DataFrame(
    signal_detections,
    columns=["horizon", "signal", "detection_idx"]
).to_csv(
    "Results/Bottleneck/adwin_signal_detections.csv",
    index=False
)


ADWIN monitoring-signal comparison:


,horizon,signal,detections,first_detection,detection_points
0,t+1,Binary_error,0,NaN,[]
1,t+1,Brier_loss,0,NaN,[]
2,t+1,Log_loss,0,NaN,[]
3,t+3,Binary_error,0,NaN,[]
4,t+3,Brier_loss,0,NaN,[]
5,t+3,Log_loss,0,NaN,[]
6,t+5,Binary_error,0,NaN,[]
7,t+5,Brier_loss,0,NaN,[]
8,t+5,Log_loss,0,NaN,[]


In [22]:
# Check existing class-weighted LSTM predictions

print("Prediction object:", type(weighted_predictions))
print("Available horizons:", weighted_predictions.keys())

for horizon in [1, 3, 5]:
    print(f"\nHorizon t+{horizon}")
    print("Columns:", weighted_predictions[horizon].columns.tolist())
    display(weighted_predictions[horizon].head())

Prediction object: <class 'dict'>
Available horizons: dict_keys([1, 3, 5])

Horizon t+1
Columns: ['idx', 'actual', 'probability', 'predicted']


,idx,actual,probability,predicted
0,400,0,0.367991,0
1,401,1,0.359980,0
2,402,0,0.242774,0
3,403,1,0.226372,0
4,404,0,0.206230,0



Horizon t+3
Columns: ['idx', 'actual', 'probability', 'predicted']


,idx,actual,probability,predicted
0,400,0,0.651014,1
1,401,1,0.595036,1
2,402,0,0.523858,1
3,403,1,0.384588,0
4,404,0,0.344070,0



Horizon t+5
Columns: ['idx', 'actual', 'probability', 'predicted']


,idx,actual,probability,predicted
0,400,0,0.615295,1
1,401,1,0.556832,1
2,402,0,0.488298,0
3,403,1,0.460683,0
4,404,0,0.618600,1


In [23]:

# STEP 17: ADWIN monitoring signal comparison
# No retraining; existing predictions only.

import os
import numpy as np
import pandas as pd
from river.drift import ADWIN

HORIZONS = [1, 3, 5]
ADWIN_DELTA = 0.002

signal_results = []
signal_detections = []

for horizon in HORIZONS:

    pred_df = weighted_predictions[horizon].copy()

    # Expected columns from the prediction dataframe
    required = ["idx", "actual", "probability", "predicted"]

    missing = [c for c in required if c not in pred_df.columns]

    if missing:
        raise ValueError(
            f"Missing columns for t+{horizon}: {missing}. "
            f"Available columns: {pred_df.columns.tolist()}"
        )

    pred_df = pred_df.sort_values("idx").reset_index(drop=True)

    y_true = pred_df["actual"].to_numpy(dtype=int)
    prob = pred_df["probability"].to_numpy(dtype=float)
    y_pred = pred_df["predicted"].to_numpy(dtype=int)
    indices = pred_df["idx"].to_numpy(dtype=int)

    # Numerical stability
    p = np.clip(prob, 1e-7, 1 - 1e-7)

    # Three monitoring signals
    signals = {
        "Binary_error": (y_true != y_pred).astype(float),

        "Brier_loss": (y_true - p) ** 2,

        "Log_loss": -(
            y_true * np.log(p)
            + (1 - y_true) * np.log(1 - p)
        )
    }

    for signal_name, values in signals.items():

        detector = ADWIN(delta=ADWIN_DELTA)
        detections = []

        for idx, value in zip(indices, values):

            detector.update(float(value))

            if detector.drift_detected:
                detections.append(int(idx))

        signal_results.append({
            "horizon": f"t+{horizon}",
            "signal": signal_name,
            "detections": len(detections),
            "first_detection": (
                detections[0] if detections else np.nan
            ),
            "detection_points": str(detections)
        })

        for idx in detections:
            signal_detections.append({
                "horizon": f"t+{horizon}",
                "signal": signal_name,
                "detection_idx": idx
            })

# Display results
signal_df = pd.DataFrame(signal_results)

print("ADWIN monitoring signal comparison")
display(signal_df)

detection_df = pd.DataFrame(
    signal_detections,
    columns=["horizon", "signal", "detection_idx"]
)

print("Individual detection points")
display(detection_df)

# Save experimental results
os.makedirs("Results/Bottleneck", exist_ok=True)

signal_df.to_csv(
    "Results/Bottleneck/adwin_signal_comparison.csv",
    index=False
)

detection_df.to_csv(
    "Results/Bottleneck/adwin_signal_detections.csv",
    index=False
)

print("Results saved successfully.")


ADWIN monitoring signal comparison


,horizon,signal,detections,first_detection,detection_points
0,t+1,Binary_error,0,NaN,[]
1,t+1,Brier_loss,0,NaN,[]
2,t+1,Log_loss,0,NaN,[]
3,t+3,Binary_error,0,NaN,[]
4,t+3,Brier_loss,0,NaN,[]
5,t+3,Log_loss,0,NaN,[]
6,t+5,Binary_error,0,NaN,[]
7,t+5,Brier_loss,0,NaN,[]
8,t+5,Log_loss,0,NaN,[]


Individual detection points


,horizon,signal,detection_idx


Results saved successfully.


In [25]:

# STEP 18: ADWIN detection timing and reliability analysis

import os
import numpy as np
import pandas as pd

# Operating phase boundaries
STABLE_START, DRIFT_START, RECOVERY_START = 400, 600, 800

# Convert detection records into a DataFrame
detections_df = pd.DataFrame(
    signal_detections,
    columns=["horizon", "signal", "detection_idx"]
)

# Assign operating phases
def assign_phase(idx):
    if idx < DRIFT_START:
        return "Stable"
    elif idx < RECOVERY_START:
        return "Drift"
    else:
        return "Recovery"

if not detections_df.empty:
    detections_df["phase"] = detections_df[
        "detection_idx"
    ].apply(assign_phase)

summary = []

for horizon in HORIZONS:

    for signal in [
        "Binary_error",
        "Brier_loss",
        "Log_loss"
    ]:

        subset = detections_df[
            (detections_df["horizon"] == horizon) &
            (detections_df["signal"] == signal)
        ].sort_values("detection_idx")

        points = subset["detection_idx"].tolist()

        false_alarms = [
            x for x in points if x < DRIFT_START
        ]

        drift_alarms = [
            x for x in points
            if DRIFT_START <= x < RECOVERY_START
        ]

        recovery_alarms = [
            x for x in points if x >= RECOVERY_START
        ]

        first_drift = (
            drift_alarms[0] if drift_alarms else np.nan
        )

        detection_delay = (
            first_drift - DRIFT_START
            if drift_alarms else np.nan
        )

        summary.append({
            "horizon": f"t+{horizon}",
            "signal": signal,
            "total_detections": len(points),
            "false_alarms_stable": len(false_alarms),
            "drift_detections": len(drift_alarms),
            "recovery_detections": len(recovery_alarms),
            "first_drift_detection": first_drift,
            "detection_delay": detection_delay,
            "all_detection_points": str(points)
        })

step18_results = pd.DataFrame(summary)

print("STEP 18: ADWIN DETECTION PERFORMANCE")
display(step18_results)

# Save results
os.makedirs("Results/Bottleneck", exist_ok=True)

step18_results.to_csv(
    "Results/Bottleneck/step18_adwin_detection_analysis.csv",
    index=False
)

detections_df.to_csv(
    "Results/Bottleneck/step18_all_detection_points.csv",
    index=False
)

print("Results saved successfully.")



STEP 18: ADWIN DETECTION PERFORMANCE


,horizon,signal,total_detections,false_alarms_stable,drift_detections,recovery_detections,first_drift_detection,detection_delay,all_detection_points
0,t+1,Binary_error,0,0,0,0,NaN,NaN,[]
1,t+1,Brier_loss,0,0,0,0,NaN,NaN,[]
2,t+1,Log_loss,0,0,0,0,NaN,NaN,[]
3,t+3,Binary_error,0,0,0,0,NaN,NaN,[]
4,t+3,Brier_loss,0,0,0,0,NaN,NaN,[]
5,t+3,Log_loss,0,0,0,0,NaN,NaN,[]
6,t+5,Binary_error,0,0,0,0,NaN,NaN,[]
7,t+5,Brier_loss,0,0,0,0,NaN,NaN,[]
8,t+5,Log_loss,0,0,0,0,NaN,NaN,[]


Results saved successfully.


In [26]:

# ============================================================
# STEP 18: ADWIN RESULTS VALIDATION
# Bottleneck Forecasting - B_V11
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

RESULTS_DIR = Path("Results/Bottleneck")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 65)
print("STEP 18: ADWIN RESULTS VALIDATION")
print("=" * 65)

# 1. Check available forecasting horizons
print("\nForecasting horizons:", HORIZONS)

# 2. Inspect weighted prediction outputs
validation_summary = []

for horizon in HORIZONS:

    print(f"\n--- Horizon t+{horizon} ---")

    if horizon not in weighted_predictions:
        print("Missing predictions!")
        continue

    df = weighted_predictions[horizon]

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head())

    # Check missing values
    missing = df.isnull().sum()

    print("\nMissing values:")
    print(missing[missing > 0])

    # Check duplicate rows
    duplicates = df.duplicated().sum()
    print("Duplicate rows:", duplicates)

    validation_summary.append({
        "Horizon": horizon,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Missing_values": int(df.isnull().sum().sum()),
        "Duplicate_rows": int(duplicates)
    })

# 3. Validate ADWIN signal outputs
print("\n--- ADWIN signal comparison ---")

print("Available variables:")
print("signal_df:", "signal_df" in globals())
print("signal_detections:", "signal_detections" in globals())

if "signal_df" in globals():
    print("\nSignal comparison shape:", signal_df.shape)
    print(signal_df.head(10))

if "signal_detections" in globals():
    print("\nDetection records:", len(signal_detections))
    print(signal_detections.head(10))

# 4. Save validation report
validation_df = pd.DataFrame(validation_summary)

validation_df.to_csv(
    RESULTS_DIR / "step18_validation_summary.csv",
    index=False
)

print("\nValidation summary:")
display(validation_df)

print("\nSaved:",
      RESULTS_DIR / "step18_validation_summary.csv")

print("\nSTEP 18 completed.")


STEP 18: ADWIN RESULTS VALIDATION

Forecasting horizons: [1, 3, 5]

--- Horizon t+1 ---
Shape: (595, 4)
Columns: ['idx', 'actual', 'probability', 'predicted']
   idx  actual  probability  predicted
0  400       0     0.367991          0
1  401       1     0.359980          0
2  402       0     0.242774          0
3  403       1     0.226372          0
4  404       0     0.206230          0

Missing values:
Series([], dtype: int64)
Duplicate rows: 0

--- Horizon t+3 ---
Shape: (595, 4)
Columns: ['idx', 'actual', 'probability', 'predicted']
   idx  actual  probability  predicted
0  400       0     0.651014          1
1  401       1     0.595036          1
2  402       0     0.523858          1
3  403       1     0.384588          0
4  404       0     0.344070          0

Missing values:
Series([], dtype: int64)
Duplicate rows: 0

--- Horizon t+5 ---
Shape: (595, 4)
Columns: ['idx', 'actual', 'probability', 'predicted']
   idx  actual  probability  predicted
0  400       0     0.615295   

AttributeError: 'list' object has no attribute 'head'

In [27]:

# ============================================================
# STEP 19: TEMPORAL DATA LEAKAGE AUDIT
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 70)
print("STEP 19: TEMPORAL DATA LEAKAGE AUDIT")
print("=" * 70)

# 1. Inspect the prepared dataset
print("\nDataset information")

if "df" in globals():
    audit_df = df.copy()
elif "df0" in globals():
    audit_df = df0.copy()
else:
    raise NameError("Could not find df or df0. Check dataset variable.")

print("Shape:", audit_df.shape)
print("Columns:", audit_df.columns.tolist())

# 2. Check time ordering
print("\nChecking temporal ordering")

print("Index monotonic:", audit_df.index.is_monotonic_increasing)

# 3. Check feature columns
print("\nFeature inspection")

if "FEATURES" in globals():
    features = FEATURES
elif "feature_cols" in globals():
    features = feature_cols
else:
    features = [
        "Processing_Time",
        "Energy_Consumption",
        "Machine_Availability"
    ]

features = [c for c in features if c in audit_df.columns]

print("Available features:", features)

# 4. Check missing values and infinite values
print("\nMissing values:")
display(audit_df[features].isnull().sum())

numeric_df = audit_df[features].select_dtypes(include=np.number)

print("\nInfinite values:")
print(np.isinf(numeric_df).sum())

# 5. Inspect temporal feature construction
print("\nChecking common future-looking column names")

future_patterns = [
    "future",
    "next",
    "lead",
    "target",
    "label",
    "t_plus",
    "t+"
]

suspicious = [
    col for col in audit_df.columns
    if any(pattern in col.lower() for pattern in future_patterns)
]

print("Potential target/future columns:", suspicious)

# 6. Check whether rolling features are shifted
print("\nIMPORTANT:")
print("Rolling features must use historical observations only.")
print("For example:")
print("df['rolling_mean'] = df['Processing_Time'].shift(1).rolling(10).mean()")

# 7. Check chronological partitions
if all(x in globals() for x in ["train_end", "drift_start",
                                 "recovery_start"]):

    print("\nTemporal boundaries:")
    print("Training end:", train_end)
    print("Drift start:", drift_start)
    print("Recovery start:", recovery_start)

# 8. Save audit report
report = {
    "dataset_rows": len(audit_df),
    "dataset_columns": len(audit_df.columns),
    "index_monotonic": audit_df.index.is_monotonic_increasing,
    "features_checked": ", ".join(features),
    "potential_future_columns": ", ".join(suspicious),
    "missing_values": int(audit_df[features].isnull().sum().sum()),
    "infinite_values": int(np.isinf(numeric_df).sum().sum())
}

report_df = pd.DataFrame([report])

output_dir = Path("Results/Bottleneck")
output_dir.mkdir(parents=True, exist_ok=True)

report_df.to_csv(
    output_dir / "step19_temporal_leakage_audit.csv",
    index=False
)

display(report_df)

print("\nSaved temporal leakage audit.")
print("STEP 19 completed.")


STEP 19: TEMPORAL DATA LEAKAGE AUDIT

Dataset information
Shape: (595, 4)
Columns: ['idx', 'actual', 'probability', 'predicted']

Checking temporal ordering
Index monotonic: True

Feature inspection
Available features: []

Missing values:


Series([], dtype: float64)


Infinite values:
Series([], dtype: float64)

Checking common future-looking column names
Potential target/future columns: []

IMPORTANT:
Rolling features must use historical observations only.
For example:
df['rolling_mean'] = df['Processing_Time'].shift(1).rolling(10).mean()


,dataset_rows,dataset_columns,index_monotonic,features_checked,potential_future_columns,missing_values,infinite_values
0,595,4,True,,,0,0



Saved temporal leakage audit.
STEP 19 completed.


In [29]:

# ============================================================
# STEP 20: ADWIN DRIFT DETECTION PERFORMANCE ANALYSIS
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 70)
print("STEP 20: ADWIN DRIFT DETECTION ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Locate ADWIN detection results
# ------------------------------------------------------------

candidate_names = [
    "signal_detections",
    "adwin_detections",
    "detections",
    "drift_points"
]

detection_data = None

for name in candidate_names:
    if name in globals():
        detection_data = globals()[name]
        print("Using detection variable:", name)
        break

if detection_data is None:
    raise NameError(
        "No ADWIN detection variable found. "
        "Inspect your existing ADWIN implementation."
    )

# Convert to DataFrame
if isinstance(detection_data, pd.DataFrame):
    detection_df = detection_data.copy()

elif isinstance(detection_data, pd.Series):
    detection_df = detection_data.to_frame()

else:
    detection_df = pd.DataFrame(detection_data)

print("\nDetection data shape:", detection_df.shape)
print("Columns:", detection_df.columns.tolist())

display(detection_df.head())

# ------------------------------------------------------------
# 2. Identify detection column
# ------------------------------------------------------------

possible_columns = [
    "drift_detected",
    "detected",
    "change_detected",
    "is_drift",
    "signal"
]

detection_col = next(
    (c for c in possible_columns if c in detection_df.columns),
    None
)

if detection_col is None:
    raise ValueError(
        "Could not identify detection indicator. "
        "Please inspect the displayed columns."
    )

# Convert to boolean
detection_values = detection_df[detection_col]

if detection_values.dtype == object:
    detection_values = detection_values.astype(str).str.lower().isin(
        ["true", "1", "yes", "drift"]
    )
else:
    detection_values = detection_values.fillna(0).astype(bool)

detection_df["ADWIN_Detected"] = detection_values

# ------------------------------------------------------------
# 3. Extract detection positions
# ------------------------------------------------------------

detection_positions = np.flatnonzero(
    detection_df["ADWIN_Detected"].to_numpy()
)

print("\nTotal ADWIN detections:", len(detection_positions))

print("Detection positions:")
print(detection_positions)

# ------------------------------------------------------------
# 4. Define operating phases
# ------------------------------------------------------------

# Original experiment boundaries:
# Stable: 400-599
# Drift: 600-799
# Recovery: 800-994

def identify_phase(position):

    if 400 <= position <= 599:
        return "Stable"

    elif 600 <= position <= 799:
        return "Drift"

    elif 800 <= position <= 994:
        return "Recovery"

    return "Outside evaluation"

detection_df["Phase"] = [
    identify_phase(i) for i in range(len(detection_df))
]

# ------------------------------------------------------------
# 5. Count detections by phase
# ------------------------------------------------------------

phase_summary = (
    detection_df.groupby("Phase")
    .agg(
        Observations=("ADWIN_Detected", "size"),
        Detections=("ADWIN_Detected", "sum")
    )
    .reset_index()
)

phase_summary["Detection_Rate"] = (
    phase_summary["Detections"] /
    phase_summary["Observations"].replace(0, np.nan)
)

print("\nADWIN phase summary:")
display(phase_summary)

# ------------------------------------------------------------
# 6. Detection positions by phase
# ------------------------------------------------------------

detected_phase = detection_df[
    detection_df["ADWIN_Detected"]
].copy()

print("\nDetected events by phase:")
display(
    detected_phase[
        ["Phase"]
    ].value_counts().rename("Detection_Count").reset_index()
)

# ------------------------------------------------------------
# 7. Detection delay
# ------------------------------------------------------------

# Reference drift onset = observation 600.
# Only calculate this if positions are in original dataset coordinates.

reference_drift_onset = 600

if len(detection_positions) > 0:

    first_detection = detection_positions[0]

    print("\nFirst detection position:", first_detection)

    if first_detection >= reference_drift_onset:
        detection_delay = (
            first_detection - reference_drift_onset
        )
        print("Detection delay:", detection_delay)
    else:
        detection_delay = np.nan
        print(
            "First detection precedes reference drift onset. "
            "Do not interpret this as detection delay."
        )

else:
    detection_delay = np.nan
    print("No detections available.")

# ------------------------------------------------------------
# 8. Save outputs
# ------------------------------------------------------------

output_dir = Path("Results/Bottleneck")
output_dir.mkdir(parents=True, exist_ok=True)

detection_df.to_csv(
    output_dir / "step20_adwin_detection_details.csv",
    index=False
)

phase_summary.to_csv(
    output_dir / "step20_adwin_phase_summary.csv",
    index=False
)

pd.DataFrame([{
    "Total_Detections": len(detection_positions),
    "First_Detection": (
        int(detection_positions[0])
        if len(detection_positions) else np.nan
    ),
    "Reference_Drift_Onset": reference_drift_onset,
    "Detection_Delay": detection_delay
}]).to_csv(
    output_dir / "step20_adwin_detection_metrics.csv",
    index=False
)

print("\nSTEP 20 completed.")
print("Results saved in:", output_dir)


STEP 20: ADWIN DRIFT DETECTION ANALYSIS
Using detection variable: signal_detections

Detection data shape: (0, 0)
Columns: []


""


ValueError: Could not identify detection indicator. Please inspect the displayed columns.

In [30]:

# ============================================================
# STEP 20A: INSPECT EXISTING ADWIN VARIABLES
# ============================================================

import pandas as pd
import numpy as np

print("=" * 65)
print("ADWIN VARIABLE INSPECTION")
print("=" * 65)

candidate_names = [
    "signal_df",
    "signal_detections",
    "adwin_detections",
    "detections",
    "drift_points",
    "weighted_predictions"
]

for name in candidate_names:

    if name in globals():

        obj = globals()[name]

        print(f"\nVariable: {name}")
        print("Type:", type(obj))

        if isinstance(obj, pd.DataFrame):

            print("Shape:", obj.shape)
            print("Columns:", obj.columns.tolist())
            display(obj.head(5))

        elif isinstance(obj, pd.Series):

            print("Shape:", obj.shape)
            display(obj.head(10))

        else:
            print("Value:", str(obj)[:1000])

# Search for other relevant variables
print("\n--- All potentially relevant variables ---")

for name in sorted(globals()):

    if any(word in name.lower() for word in
           ["adwin", "drift", "signal", "detect", "error"]):

        print(name, type(globals()[name]).__name__)


ADWIN VARIABLE INSPECTION

Variable: signal_df
Type: <class 'pandas.DataFrame'>
Shape: (9, 5)
Columns: ['horizon', 'signal', 'detections', 'first_detection', 'detection_points']


,horizon,signal,detections,first_detection,detection_points
0,t+1,Binary_error,0,NaN,[]
1,t+1,Brier_loss,0,NaN,[]
2,t+1,Log_loss,0,NaN,[]
3,t+3,Binary_error,0,NaN,[]
4,t+3,Brier_loss,0,NaN,[]



Variable: signal_detections
Type: <class 'list'>
Value: []

Variable: detections
Type: <class 'list'>
Value: []

Variable: weighted_predictions
Type: <class 'dict'>
Value: {1:      idx  actual  probability  predicted
0    400       0     0.367991          0
1    401       1     0.359980          0
2    402       0     0.242774          0
3    403       1     0.226372          0
4    404       0     0.206230          0
..   ...     ...          ...        ...
590  990       1     0.496214          0
591  991       1     0.381585          0
592  992       0     0.408658          0
593  993       0     0.477312          0
594  994       0     0.520384          1

[595 rows x 4 columns], 3:      idx  actual  probability  predicted
0    400       0     0.651014          1
1    401       1     0.595036          1
2    402       0     0.523858          1
3    403       1     0.384588          0
4    404       0     0.344070          0
..   ...     ...          ...        ...
590  990       1

In [31]:

# ============================================================
# STEP 20B: CORRECTED ADWIN ANALYSIS
# Uses the actual signal_df structure
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 70)
print("STEP 20B: ADWIN DETECTION ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Validate ADWIN results
# ------------------------------------------------------------

required = [
    "horizon", "signal", "detections",
    "first_detection", "detection_points"
]

missing = [c for c in required if c not in signal_df.columns]

if missing:
    raise ValueError(f"Missing columns in signal_df: {missing}")

adwin_results = signal_df.copy()

adwin_results["detections"] = (
    pd.to_numeric(adwin_results["detections"], errors="coerce")
    .fillna(0).astype(int)
)

print("\nOriginal ADWIN results:")
display(adwin_results)

print("\nTotal monitoring configurations:",
      len(adwin_results))

print("Total detections:",
      adwin_results["detections"].sum())

# ------------------------------------------------------------
# 2. Expand detection points
# ------------------------------------------------------------

def normalize_points(value):

    if isinstance(value, (list, tuple, np.ndarray)):
        return list(value)

    if value is None:
        return []

    if isinstance(value, float) and np.isnan(value):
        return []

    return [value]


expanded = []

for _, row in adwin_results.iterrows():

    points = normalize_points(row["detection_points"])

    for point in points:

        expanded.append({
            "Horizon": row["horizon"],
            "Signal": row["signal"],
            "Detection_Position": int(point)
        })

detection_details = pd.DataFrame(
    expanded,
    columns=["Horizon", "Signal", "Detection_Position"]
)

print("\nExpanded detection events:")
display(detection_details)

# ------------------------------------------------------------
# 3. Operating phase classification
# Original dataset positions:
# Stable: 400-599
# Drift: 600-799
# Recovery: 800-994
# ------------------------------------------------------------

def get_phase(position):

    if 400 <= position <= 599:
        return "Stable"

    if 600 <= position <= 799:
        return "Drift"

    if 800 <= position <= 994:
        return "Recovery"

    return "Outside evaluation"


if not detection_details.empty:

    detection_details["Phase"] = (
        detection_details["Detection_Position"]
        .apply(get_phase)
    )

# ------------------------------------------------------------
# 4. Summarize detections by horizon and signal
# ------------------------------------------------------------

phase_order = ["Stable", "Drift", "Recovery"]

summary = []

for _, row in adwin_results.iterrows():

    points = normalize_points(row["detection_points"])

    phase_counts = {
        phase: sum(
            get_phase(int(p)) == phase for p in points
        )
        for phase in phase_order
    }

    summary.append({
        "Horizon": row["horizon"],
        "Signal": row["signal"],
        "Total_Detections": len(points),
        "Stable_Detections": phase_counts["Stable"],
        "Drift_Detections": phase_counts["Drift"],
        "Recovery_Detections": phase_counts["Recovery"],
        "First_Detection": (
            min(points) if points else np.nan
        )
    })

adwin_summary = pd.DataFrame(summary)

print("\nADWIN detection summary:")
display(adwin_summary)

# ------------------------------------------------------------
# 5. Check prediction coverage by operating phase
# ------------------------------------------------------------

coverage = []

for horizon, predictions in weighted_predictions.items():

    temp = predictions.copy()

    temp["Phase"] = temp["idx"].apply(get_phase)

    for phase in phase_order:

        phase_data = temp[temp["Phase"] == phase]

        coverage.append({
            "Horizon": f"t+{horizon}",
            "Phase": phase,
            "Observations": len(phase_data),
            "Actual_Positive": int(
                phase_data["actual"].sum()
            ),
            "Predicted_Positive": int(
                phase_data["predicted"].sum()
            )
        })

coverage_df = pd.DataFrame(coverage)

print("\nPrediction coverage:")
display(coverage_df)

# ------------------------------------------------------------
# 6. Detection delay
# ------------------------------------------------------------

reference_onset = 600

delay_records = []

for _, row in adwin_results.iterrows():

    points = normalize_points(row["detection_points"])

    valid_points = [
        int(p) for p in points
        if int(p) >= reference_onset
    ]

    delay_records.append({
        "Horizon": row["horizon"],
        "Signal": row["signal"],
        "Detection_Delay": (
            min(valid_points) - reference_onset
            if valid_points else np.nan
        ),
        "Detection_Available": bool(valid_points)
    })

delay_df = pd.DataFrame(delay_records)

print("\nDetection delay analysis:")
display(delay_df)

# ------------------------------------------------------------
# 7. Save all outputs
# ------------------------------------------------------------

output_dir = Path("Results/Bottleneck")
output_dir.mkdir(parents=True, exist_ok=True)

adwin_results.to_csv(
    output_dir / "step20_adwin_original_results.csv",
    index=False
)

detection_details.to_csv(
    output_dir / "step20_adwin_detection_details.csv",
    index=False
)

adwin_summary.to_csv(
    output_dir / "step20_adwin_phase_summary.csv",
    index=False
)

coverage_df.to_csv(
    output_dir / "step20_prediction_phase_coverage.csv",
    index=False
)

delay_df.to_csv(
    output_dir / "step20_detection_delay.csv",
    index=False
)

# ------------------------------------------------------------
# 8. Final interpretation
# ------------------------------------------------------------

total = adwin_results["detections"].sum()

print("\n" + "=" * 70)
print("STEP 20 CONCLUSION")
print("=" * 70)

if total == 0:

    print("ADWIN generated zero detections across all configurations.")
    print("Detection delay cannot be estimated.")
    print("Detection sensitivity cannot be evaluated from these alarms.")
    print("Further investigation of the monitoring stream is required.")

else:

    print("ADWIN generated", total, "detections.")
    print("Inspect phase summaries and detection positions.")

print("\nAll Step 20 outputs saved to:", output_dir)
print("STEP 20B COMPLETED")


STEP 20B: ADWIN DETECTION ANALYSIS

Original ADWIN results:


,horizon,signal,detections,first_detection,detection_points
0,t+1,Binary_error,0,NaN,[]
1,t+1,Brier_loss,0,NaN,[]
2,t+1,Log_loss,0,NaN,[]
3,t+3,Binary_error,0,NaN,[]
4,t+3,Brier_loss,0,NaN,[]
5,t+3,Log_loss,0,NaN,[]
6,t+5,Binary_error,0,NaN,[]
7,t+5,Brier_loss,0,NaN,[]
8,t+5,Log_loss,0,NaN,[]



Total monitoring configurations: 9
Total detections: 0


ValueError: invalid literal for int() with base 10: '[]'

In [32]:

# ============================================================
# STEP 20C: ROBUST ADWIN DETECTION ANALYSIS
# Handles lists, empty lists and string representations
# ============================================================

import ast
import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 70)
print("STEP 20C: ROBUST ADWIN ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Robust detection-point parser
# ------------------------------------------------------------

def normalize_points(value):

    if value is None:
        return []

    if isinstance(value, float) and np.isnan(value):
        return []

    if isinstance(value, str):

        value = value.strip()

        if value in ("", "[]", "None", "nan"):
            return []

        try:
            value = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return [int(value)]

    if isinstance(value, (list, tuple, np.ndarray)):
        return [
            int(x) for x in value
            if str(x).strip() not in ("", "None")
        ]

    return [int(value)]


# ------------------------------------------------------------
# 2. Normalize original ADWIN results
# ------------------------------------------------------------

adwin_results = signal_df.copy()

adwin_results["detection_points"] = (
    adwin_results["detection_points"]
    .apply(normalize_points)
)

adwin_results["detections"] = (
    adwin_results["detection_points"].apply(len)
)

print("\nValidated ADWIN results:")
display(adwin_results)

print(
    "\nTotal detections:",
    adwin_results["detections"].sum()
)

# ------------------------------------------------------------
# 3. Expand detection events
# ------------------------------------------------------------

expanded = []

for _, row in adwin_results.iterrows():

    for point in row["detection_points"]:

        expanded.append({
            "Horizon": row["horizon"],
            "Signal": row["signal"],
            "Detection_Position": point
        })

detection_details = pd.DataFrame(
    expanded,
    columns=[
        "Horizon",
        "Signal",
        "Detection_Position"
    ]
)

# ------------------------------------------------------------
# 4. Define operating phases
# ------------------------------------------------------------

def get_phase(position):

    if 400 <= position <= 599:
        return "Stable"

    elif 600 <= position <= 799:
        return "Drift"

    elif 800 <= position <= 994:
        return "Recovery"

    return "Outside evaluation"


if not detection_details.empty:

    detection_details["Phase"] = (
        detection_details["Detection_Position"]
        .apply(get_phase)
    )

print("\nDetection events:")
display(detection_details)

# ------------------------------------------------------------
# 5. Phase-wise detection summary
# ------------------------------------------------------------

phase_summary = []

for _, row in adwin_results.iterrows():

    points = row["detection_points"]

    phase_summary.append({
        "Horizon": row["horizon"],
        "Signal": row["signal"],
        "Total_Detections": len(points),
        "Stable_Detections": sum(
            get_phase(p) == "Stable" for p in points
        ),
        "Drift_Detections": sum(
            get_phase(p) == "Drift" for p in points
        ),
        "Recovery_Detections": sum(
            get_phase(p) == "Recovery" for p in points
        )
    })

phase_df = pd.DataFrame(phase_summary)

print("\nPhase-wise ADWIN summary:")
display(phase_df)

# ------------------------------------------------------------
# 6. Prediction coverage
# ------------------------------------------------------------

coverage = []

for horizon, predictions in weighted_predictions.items():

    temp = predictions.copy()

    temp["Phase"] = temp["idx"].apply(get_phase)

    for phase in ["Stable", "Drift", "Recovery"]:

        subset = temp[temp["Phase"] == phase]

        coverage.append({
            "Horizon": f"t+{horizon}",
            "Phase": phase,
            "Observations": len(subset),
            "Actual_Positive": int(subset["actual"].sum()),
            "Predicted_Positive": int(
                subset["predicted"].sum()
            )
        })

coverage_df = pd.DataFrame(coverage)

print("\nPrediction phase coverage:")
display(coverage_df)

# ------------------------------------------------------------
# 7. Detection delay
# ------------------------------------------------------------

REFERENCE_DRIFT_START = 600

delay_results = []

for _, row in adwin_results.iterrows():

    points = row["detection_points"]

    post_drift = [
        p for p in points
        if p >= REFERENCE_DRIFT_START
    ]

    delay_results.append({
        "Horizon": row["horizon"],
        "Signal": row["signal"],
        "First_Post_Drift_Detection": (
            min(post_drift) if post_drift else np.nan
        ),
        "Detection_Delay": (
            min(post_drift) - REFERENCE_DRIFT_START
            if post_drift else np.nan
        )
    })

delay_df = pd.DataFrame(delay_results)

print("\nDetection delay:")
display(delay_df)

# ------------------------------------------------------------
# 8. Save outputs
# ------------------------------------------------------------

output_dir = Path("Results/Bottleneck")
output_dir.mkdir(parents=True, exist_ok=True)

adwin_results.to_csv(
    output_dir / "step20_adwin_results.csv",
    index=False
)

detection_details.to_csv(
    output_dir / "step20_detection_details.csv",
    index=False
)

phase_df.to_csv(
    output_dir / "step20_phase_summary.csv",
    index=False
)

coverage_df.to_csv(
    output_dir / "step20_prediction_coverage.csv",
    index=False
)

delay_df.to_csv(
    output_dir / "step20_detection_delay.csv",
    index=False
)

# ------------------------------------------------------------
# 9. Final status
# ------------------------------------------------------------

total_detections = int(
    adwin_results["detections"].sum()
)

print("\n" + "=" * 70)

if total_detections == 0:

    print("RESULT: ZERO ADWIN DETECTIONS")
    print("Detection delay: Not estimable.")
    print("Sensitivity: Not estimable from current alarms.")

else:

    print("RESULT:", total_detections, "ADWIN detections")

print("All outputs saved:", output_dir)
print("STEP 20C COMPLETED")


STEP 20C: ROBUST ADWIN ANALYSIS

Validated ADWIN results:


,horizon,signal,detections,first_detection,detection_points
0,t+1,Binary_error,0,NaN,[]
1,t+1,Brier_loss,0,NaN,[]
2,t+1,Log_loss,0,NaN,[]
3,t+3,Binary_error,0,NaN,[]
4,t+3,Brier_loss,0,NaN,[]
5,t+3,Log_loss,0,NaN,[]
6,t+5,Binary_error,0,NaN,[]
7,t+5,Brier_loss,0,NaN,[]
8,t+5,Log_loss,0,NaN,[]



Total detections: 0

Detection events:


,Horizon,Signal,Detection_Position



Phase-wise ADWIN summary:


,Horizon,Signal,Total_Detections,Stable_Detections,Drift_Detections,Recovery_Detections
0,t+1,Binary_error,0,0,0,0
1,t+1,Brier_loss,0,0,0,0
2,t+1,Log_loss,0,0,0,0
3,t+3,Binary_error,0,0,0,0
4,t+3,Brier_loss,0,0,0,0
5,t+3,Log_loss,0,0,0,0
6,t+5,Binary_error,0,0,0,0
7,t+5,Brier_loss,0,0,0,0
8,t+5,Log_loss,0,0,0,0



Prediction phase coverage:


,Horizon,Phase,Observations,Actual_Positive,Predicted_Positive
0,t+1,Stable,200,63,107
1,t+1,Drift,200,89,133
2,t+1,Recovery,195,56,78
3,t+3,Stable,200,63,100
4,t+3,Drift,200,89,103
5,t+3,Recovery,195,56,65
6,t+5,Stable,200,63,94
7,t+5,Drift,200,89,147
8,t+5,Recovery,195,56,71



Detection delay:


,Horizon,Signal,First_Post_Drift_Detection,Detection_Delay
0,t+1,Binary_error,NaN,NaN
1,t+1,Brier_loss,NaN,NaN
2,t+1,Log_loss,NaN,NaN
3,t+3,Binary_error,NaN,NaN
4,t+3,Brier_loss,NaN,NaN
5,t+3,Log_loss,NaN,NaN
6,t+5,Binary_error,NaN,NaN
7,t+5,Brier_loss,NaN,NaN
8,t+5,Log_loss,NaN,NaN



RESULT: ZERO ADWIN DETECTIONS
Detection delay: Not estimable.
Sensitivity: Not estimable from current alarms.
All outputs saved: Results\Bottleneck
STEP 20C COMPLETED


In [33]:

# ============================================================
# STEP 21: ADWIN MONITORING DIAGNOSTICS AND SENSITIVITY
# Original experiment remains unchanged
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 75)
print("STEP 21: ADWIN MONITORING DIAGNOSTICS")
print("=" * 75)

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

OUTPUT_DIR = Path("Results/Bottleneck")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DELTAS = [0.1, 0.05, 0.01, 0.005, 0.001]

PHASES = {
    "Stable": (400, 599),
    "Drift": (600, 799),
    "Recovery": (800, 994)
}

# Use the existing ADWIN implementation.
ADWIN_CLASS = type(detector)

print("ADWIN implementation:", ADWIN_CLASS)
print("Original ADWIN delta:", ADWIN_DELTA)
print("Sensitivity values:", DELTAS)

# ------------------------------------------------------------
# 2. Reconstruct monitoring streams
# ------------------------------------------------------------

monitoring_streams = {}

for horizon, predictions in weighted_predictions.items():

    temp = predictions.sort_values("idx").copy()

    required_columns = [
        "idx", "actual", "probability", "predicted"
    ]

    missing = [
        c for c in required_columns if c not in temp.columns
    ]

    if missing:
        raise ValueError(
            f"Missing prediction columns for t+{horizon}: {missing}"
        )

    y = temp["actual"].to_numpy(dtype=int)
    p = temp["probability"].to_numpy(dtype=float)
    predicted = temp["predicted"].to_numpy(dtype=int)

    if not np.isfinite(p).all():
        raise ValueError(f"Invalid probabilities for t+{horizon}")

    if ((p < 0) | (p > 1)).any():
        raise ValueError(f"Probabilities outside [0,1] for t+{horizon}")

    p_clip = np.clip(p, 1e-7, 1 - 1e-7)

    temp["Binary_error"] = (y != predicted).astype(float)

    temp["Brier_loss"] = (y - p) ** 2

    temp["Log_loss"] = -(
        y * np.log(p_clip)
        + (1 - y) * np.log(1 - p_clip)
    )

    monitoring_streams[horizon] = temp

print("\nMonitoring streams reconstructed successfully.")

# ------------------------------------------------------------
# 3. Compare error behaviour across phases
# ------------------------------------------------------------

phase_records = []

for horizon, temp in monitoring_streams.items():

    for phase, (start, end) in PHASES.items():

        subset = temp[
            temp["idx"].between(start, end)
        ]

        for signal in [
            "Binary_error",
            "Brier_loss",
            "Log_loss"
        ]:

            values = subset[signal]

            phase_records.append({
                "Horizon": f"t+{horizon}",
                "Phase": phase,
                "Signal": signal,
                "N": len(values),
                "Mean": values.mean(),
                "Std": values.std(),
                "Minimum": values.min(),
                "Maximum": values.max()
            })

phase_diagnostics = pd.DataFrame(phase_records)

print("\nPHASE-WISE ERROR ANALYSIS")
display(phase_diagnostics.round(5))

# ------------------------------------------------------------
# 4. Test ADWIN sensitivity
# ------------------------------------------------------------

sensitivity_records = []

for horizon, temp in monitoring_streams.items():

    temp = temp.sort_values("idx").reset_index(drop=True)

    for signal in [
        "Binary_error",
        "Brier_loss",
        "Log_loss"
    ]:

        values = temp[signal].to_numpy()
        indices = temp["idx"].to_numpy()

        for delta in DELTAS:

            try:

                adwin = ADWIN_CLASS(delta=delta)

                alarm_positions = []

                for position, value in zip(indices, values):

                    adwin.update(float(value))

                    # River ADWIN detection property
                    alarm = getattr(
                        adwin,
                        "drift_detected",
                        getattr(adwin, "change_detected", False)
                    )

                    if alarm:
                        alarm_positions.append(int(position))

                sensitivity_records.append({
                    "Horizon": f"t+{horizon}",
                    "Signal": signal,
                    "Delta": delta,
                    "Total_Detections": len(alarm_positions),
                    "Detection_Points": alarm_positions,
                    "First_Detection": (
                        alarm_positions[0]
                        if alarm_positions else np.nan
                    )
                })

            except Exception as exc:

                print(
                    f"ADWIN failed: horizon={horizon}, "
                    f"signal={signal}, delta={delta}: {exc}"
                )

sensitivity_df = pd.DataFrame(sensitivity_records)

print("\nADWIN SENSITIVITY RESULTS")
display(sensitivity_df)

# ------------------------------------------------------------
# 5. Summarize detections by operating phase
# ------------------------------------------------------------

phase_detection_records = []

for _, row in sensitivity_df.iterrows():

    points = row["Detection_Points"]

    phase_detection_records.append({
        "Horizon": row["Horizon"],
        "Signal": row["Signal"],
        "Delta": row["Delta"],
        "Stable": sum(400 <= p <= 599 for p in points),
        "Drift": sum(600 <= p <= 799 for p in points),
        "Recovery": sum(800 <= p <= 994 for p in points)
    })

phase_detection_df = pd.DataFrame(
    phase_detection_records
)

print("\nDETECTIONS BY OPERATING PHASE")
display(phase_detection_df)

# ------------------------------------------------------------
# 6. Save monitoring streams and diagnostic results
# ------------------------------------------------------------

all_streams = []

for horizon, temp in monitoring_streams.items():

    saved = temp.copy()
    saved.insert(0, "Horizon", f"t+{horizon}")

    all_streams.append(saved)

streams_df = pd.concat(all_streams, ignore_index=True)

streams_df.to_csv(
    OUTPUT_DIR / "step21_monitoring_streams.csv",
    index=False
)

phase_diagnostics.to_csv(
    OUTPUT_DIR / "step21_phase_error_diagnostics.csv",
    index=False
)

sensitivity_df.to_csv(
    OUTPUT_DIR / "step21_adwin_sensitivity.csv",
    index=False
)

phase_detection_df.to_csv(
    OUTPUT_DIR / "step21_adwin_phase_detections.csv",
    index=False
)

# ------------------------------------------------------------
# 7. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 21 COMPLETED")
print("=" * 75)

print("Original experiments preserved.")
print("Monitoring streams:", len(streams_df))
print("Sensitivity configurations:", len(sensitivity_df))

if not sensitivity_df.empty:

    print(
        "Total detections across sensitivity tests:",
        sensitivity_df["Total_Detections"].sum()
    )

print("\nFiles saved in:", OUTPUT_DIR)


STEP 21: ADWIN MONITORING DIAGNOSTICS
ADWIN implementation: <class 'river.drift.adwin.ADWIN'>
Original ADWIN delta: 0.002
Sensitivity values: [0.1, 0.05, 0.01, 0.005, 0.001]

Monitoring streams reconstructed successfully.

PHASE-WISE ERROR ANALYSIS


,Horizon,Phase,Signal,N,Mean,Std,Minimum,Maximum
0,t+1,Stable,Binary_error,200,0.53000,0.50035,0.00000,1.00000
1,t+1,Stable,Brier_loss,200,0.26309,0.11647,0.03727,0.63911
2,t+1,Stable,Log_loss,200,0.72073,0.24794,0.21450,1.60665
3,t+1,Drift,Binary_error,200,0.46000,0.49965,0.00000,1.00000
4,t+1,Drift,Brier_loss,200,0.26651,0.14113,0.02807,0.70257
5,t+1,Drift,Log_loss,200,0.73064,0.30268,0.18338,1.82135
6,t+1,Recovery,Binary_error,195,0.43077,0.49646,0.00000,1.00000
7,t+1,Recovery,Brier_loss,195,0.25724,0.14717,0.03579,0.70006
8,t+1,Recovery,Log_loss,195,0.71164,0.31998,0.20970,1.81214
9,t+3,Stable,Binary_error,200,0.47500,0.50063,0.00000,1.00000



ADWIN SENSITIVITY RESULTS


,Horizon,Signal,Delta,Total_Detections,Detection_Points,First_Detection
0,t+1,Binary_error,0.100,0,[],NaN
1,t+1,Binary_error,0.050,0,[],NaN
2,t+1,Binary_error,0.010,0,[],NaN
3,t+1,Binary_error,0.005,0,[],NaN
4,t+1,Binary_error,0.001,0,[],NaN
5,t+1,Brier_loss,0.100,0,[],NaN
6,t+1,Brier_loss,0.050,0,[],NaN
7,t+1,Brier_loss,0.010,0,[],NaN
8,t+1,Brier_loss,0.005,0,[],NaN
9,t+1,Brier_loss,0.001,0,[],NaN



DETECTIONS BY OPERATING PHASE


,Horizon,Signal,Delta,Stable,Drift,Recovery
0,t+1,Binary_error,0.100,0,0,0
1,t+1,Binary_error,0.050,0,0,0
2,t+1,Binary_error,0.010,0,0,0
3,t+1,Binary_error,0.005,0,0,0
4,t+1,Binary_error,0.001,0,0,0
5,t+1,Brier_loss,0.100,0,0,0
6,t+1,Brier_loss,0.050,0,0,0
7,t+1,Brier_loss,0.010,0,0,0
8,t+1,Brier_loss,0.005,0,0,0
9,t+1,Brier_loss,0.001,0,0,0



STEP 21 COMPLETED
Original experiments preserved.
Monitoring streams: 1785
Sensitivity configurations: 45
Total detections across sensitivity tests: 0

Files saved in: Results\Bottleneck


In [19]:
# Data source : https://www.kaggle.com/datasets/ziya07/manufacturing-production-data?utm_source=chatgpt.com